In [1]:
from classes.neo_4j import Neo4j
from pyspark.sql import SparkSession
from neo4j import GraphDatabase
from pyspark.sql.types import StructType, StructField, StringType
from pyspark.sql.functions import col 

Neo4j.create_connection("neo4j+s://5c69cfe8.databases.neo4j.io", "neo4j", "Zt6QrdRSYTWDGXN1RgGiCAvDUCKX1u9Rh7TPwprdoOQ" )

In [2]:
file_path = "file:///home/student/de-assgt/content/final_dictionary"

spark = SparkSession.builder.appName('StreamCSVToNeo4j').getOrCreate() 

df_sample = spark.read.option("header", "true").option("delimiter", "\t").csv(file_path)

column_names = [col_name for col_name, dtype in df_sample.dtypes]

schema = StructType([
    StructField(col_name, StringType(), True) 
    for col_name in column_names  
])



24/12/22 20:29:56 WARN Utils: Your hostname, LAPTOP-PFPL3CLD. resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
24/12/22 20:29:56 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/12/22 20:29:56 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
df = spark.readStream.option("header","true").option("delimiter", "\t").schema(schema).csv(file_path)


In [4]:
# Define which columns should become nodes and relationships
node = ['words', 'kata_terbitan', 'pos_tag', 'word_length'] 
relationship = [
    ('words', 'kata_terbitan', 'CAN_BE'),
    ('words', 'pos_tag', 'IS_A'),
    ('words', 'word_length', 'HAS_A')
]
properties = {
    'words': ['definition', 'kata_dasar', 'entity_type', 'sentiment'],
    'pos_tag': ['freq_of_use'],
    'kata_terbitan': ['num_variations'],
    'word_length': ['length_category']
} 

In [5]:
# Start the query
query = df.writeStream \
    .outputMode("update") \
    .foreachBatch(lambda df, epoch_id: Neo4j.write_to_neo4j(df, epoch_id, node, relationship, properties)) \
    .start()


24/12/22 20:30:02 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-8a38f465-1afc-4a53-8c8e-ef6fa9e5443e. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
24/12/22 20:30:02 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


In [6]:
Neo4j.close_connection()

In [7]:
spark.stop()